# MambaIRv2 TPAF Denoising — Inference & Evaluation Notebook (Phases 5–6, 8)

This notebook covers:

- **Paired evaluation** on validation data with PSNR & SSIM metrics (Phase 5–6)
- **Standalone inference** on new unpaired noisy images from Drive (Phase 6.2)
- **Tiled inference** fallback for OOM situations (Phase 6.3)
- **Visual side-by-side comparison**: noisy / denoised / ground truth (matplotlib)
- **Aggregate metrics** computation and results saved back to Drive (Phase 8)

## 0  User Configuration
Edit **only this cell** before running the notebook.

In [ ]:
# ============================================================
# USER CONFIGURATION — edit these values before running
# ============================================================

# --- Google Drive paths ---
DRIVE_DATA_ROOT   = "/content/drive/MyDrive/TPAF_data"
# ^ Must contain:  val/clean/  val/noisy/

DRIVE_EXPR_ROOT   = "/content/drive/MyDrive/MambaIR_experiments"
# ^ Where training saved checkpoints (same as train.ipynb setting)

DRIVE_NOISY_DIR   = "/content/drive/MyDrive/TPAF_new_noisy"
# ^ New unpaired noisy images for standalone inference (can be same as val/noisy)

DRIVE_RESULTS_DIR = "/content/drive/MyDrive/TPAF_results"
# ^ Where denoised outputs and metrics will be saved

# --- Checkpoint ---
CHECKPOINT_PATH   = None
# Set to None to auto-select net_g_latest.pth from DRIVE_EXPR_ROOT.
# Or specify a path: "/content/drive/MyDrive/MambaIR_experiments/MambaIRv2_TPAF_Denoise/models/net_g_400000.pth"

# --- Local (Colab) paths ---
LOCAL_DATA_ROOT   = "/content/data"
LOCAL_NOISY_DIR   = "/content/new_noisy"   # copied from DRIVE_NOISY_DIR
LOCAL_RESULTS_DIR = "/content/results"
MAMBAIR_DIR       = "/content/MambaIR"

# --- Tiled inference (OOM fallback) ---
TILE_SIZE    = 256   # tile side length in pixels (must be divisible by window_size=16)
TILE_OVERLAP = 32    # overlap between adjacent tiles (pixels)

# --- Visualisation ---
VIS_NUM_IMAGES = 5   # how many val images to show in the side-by-side comparison

# --- Model architecture (must match what was used during training) ---
EMBED_DIM = 180
DEPTHS    = [6, 6, 6, 6, 6, 6]
NUM_HEADS = [6, 6, 6, 6, 6, 6]

# ============================================================
print("Configuration loaded.")
print(f"  Drive data root   : {DRIVE_DATA_ROOT}")
print(f"  Drive expr root   : {DRIVE_EXPR_ROOT}")
print(f"  Drive noisy dir   : {DRIVE_NOISY_DIR}")
print(f"  Drive results dir : {DRIVE_RESULTS_DIR}")
print(f"  Checkpoint        : {CHECKPOINT_PATH or 'auto-detect'}")

## 1  Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

## 2  Clone MambaIR Repository & Install Dependencies

> `mamba_ssm` compilation takes ~3–5 minutes on first run.

In [ ]:
import os, subprocess, sys

# Clone repo
if not os.path.exists(MAMBAIR_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/csguoh/MambaIR.git", MAMBAIR_DIR],
        check=True,
    )
    print(f"Cloned MambaIR to {MAMBAIR_DIR}")
else:
    print(f"Repo already exists at {MAMBAIR_DIR} — skipping clone.")

os.chdir(MAMBAIR_DIR)

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("torch==2.0.1", "torchvision==0.15.2", "torchaudio==2.0.2",
    "--index-url", "https://download.pytorch.org/whl/cu118")

try:
    pip("causal_conv1d==1.1.1")
    pip("mamba_ssm==1.1.1")
    print("mamba_ssm 1.1.1 installed.")
except Exception:
    print("Fixed-version install failed — trying latest from source...")
    pip("causal-conv1d>=1.1.0")
    pip("mamba-ssm", "--no-build-isolation")

pip("basicsr==1.3.5", "einops", "timm")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
pip("scikit-image")   # for PSNR / SSIM in Phase 8

print("\nAll dependencies installed.")

## 3  Copy Data & New Noisy Images from Drive to Local Storage

In [ ]:
import shutil, os

LOCAL_VAL_CLEAN = f"{LOCAL_DATA_ROOT}/val/clean"
LOCAL_VAL_NOISY = f"{LOCAL_DATA_ROOT}/val/noisy"

# Validation data (for paired evaluation)
for src, dst in [
    (f"{DRIVE_DATA_ROOT}/val/clean", LOCAL_VAL_CLEAN),
    (f"{DRIVE_DATA_ROOT}/val/noisy", LOCAL_VAL_NOISY),
]:
    if os.path.exists(dst):
        print(f"Already exists — skipping: {dst}")
    else:
        print(f"Copying {src}  →  {dst} ...")
        shutil.copytree(src, dst)

# New unpaired noisy images
if os.path.exists(DRIVE_NOISY_DIR):
    if os.path.exists(LOCAL_NOISY_DIR):
        print(f"Already exists — skipping: {LOCAL_NOISY_DIR}")
    else:
        print(f"Copying {DRIVE_NOISY_DIR}  →  {LOCAL_NOISY_DIR} ...")
        shutil.copytree(DRIVE_NOISY_DIR, LOCAL_NOISY_DIR)
else:
    print(f"DRIVE_NOISY_DIR not found ({DRIVE_NOISY_DIR}) — standalone inference will be skipped.")
    LOCAL_NOISY_DIR = None

os.makedirs(LOCAL_RESULTS_DIR, exist_ok=True)
print("\nData ready.")

## 4  Auto-detect `in_chans`

Loads one validation sample through BasicSR's pipeline to discover the real channel count.

In [ ]:
import sys
sys.path.insert(0, MAMBAIR_DIR)
from basicsr.data.paired_image_dataset import PairedImageDataset

_opt = {
    "dataroot_gt"   : LOCAL_VAL_CLEAN,
    "dataroot_lq"   : LOCAL_VAL_NOISY,
    "filename_tmpl" : "{}",
    "io_backend"    : {"type": "disk"},
    "gt_size"       : 128,
    "use_hflip"     : False,
    "use_rot"       : False,
    "phase"         : "train",
    "scale"         : 1,
}
_ds     = PairedImageDataset(_opt)
_sample = _ds[0]
IN_CHANS = int(_sample["gt"].shape[0])
del _ds, _sample, _opt

print(f"Detected in_chans = {IN_CHANS}")
if IN_CHANS == 1:
    print("  → Grayscale kept as single-channel.")
else:
    print("  → BasicSR converted grayscale to 3-channel RGB.")

## 5  Auto-detect `model_type`

In [ ]:
import glob, re

MODEL_TYPE = None
for pattern in [
    f"{MAMBAIR_DIR}/options/train/mambairv2/*DN*.yml",
    f"{MAMBAIR_DIR}/options/train/mambairv2/*denois*.yml",
    f"{MAMBAIR_DIR}/options/train/mambairv2/*.yml",
]:
    for yml_path in sorted(glob.glob(pattern)):
        with open(yml_path) as fh:
            text = fh.read()
        m = re.search(r'^model_type:\s*(\S+)', text, re.MULTILINE)
        if m:
            MODEL_TYPE = m.group(1)
            print(f"Found model_type '{MODEL_TYPE}'  (from {yml_path})")
            break
    if MODEL_TYPE:
        break

if MODEL_TYPE is None:
    MODEL_TYPE = "ImageCleanModel"
    print(f"Defaulting to model_type = '{MODEL_TYPE}'")

print(f"\nUsing model_type = {MODEL_TYPE}")

## 6  Resolve Checkpoint Path

If `CHECKPOINT_PATH` is `None`, the notebook auto-selects `net_g_latest.pth` (or the highest-numbered checkpoint) from your Drive experiment directory.

In [ ]:
import glob, os

EXPR_MODELS_DIR = f"{DRIVE_EXPR_ROOT}/MambaIRv2_TPAF_Denoise/models"

if CHECKPOINT_PATH is None:
    latest_path = f"{EXPR_MODELS_DIR}/net_g_latest.pth"
    if os.path.exists(latest_path):
        CHECKPOINT_PATH = latest_path
    else:
        candidates = sorted(glob.glob(f"{EXPR_MODELS_DIR}/net_g_*.pth"))
        assert candidates, (
            f"No checkpoints found in {EXPR_MODELS_DIR}.\n"
            "Train the model first (train.ipynb) or set CHECKPOINT_PATH manually.")
        CHECKPOINT_PATH = candidates[-1]
        print(f"net_g_latest.pth not found — using: {os.path.basename(CHECKPOINT_PATH)}")

assert os.path.exists(CHECKPOINT_PATH), f"Checkpoint not found: {CHECKPOINT_PATH}"
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"  Size: {os.path.getsize(CHECKPOINT_PATH) / 1e6:.1f} MB")

## 7  Generate Test YAML Config

At test time, `img_size=512` (full image resolution).
`window_size=16` divides 512 evenly (512/16 = 32), so no padding is needed.

In [ ]:
import os

yml_dir       = f"{MAMBAIR_DIR}/options/test/mambairv2"
TEST_YML_PATH = f"{yml_dir}/test_MambaIRv2_TPAF_denoise.yml"
os.makedirs(yml_dir, exist_ok=True)

TEST_RESULTS_DIR = f"{LOCAL_RESULTS_DIR}/paired_eval"
os.makedirs(TEST_RESULTS_DIR, exist_ok=True)

TEST_YML = f"""# -----------------------------------------------------------------------
# MambaIRv2 Denoising Test — TPAF Microscopy (auto-generated)
# -----------------------------------------------------------------------
name: MambaIRv2_TPAF_Denoise_Test
model_type: {MODEL_TYPE}
scale: 1
num_gpu: 1
manual_seed: 42

datasets:
  test_1:
    task: DN
    name: TPAF_val
    type: PairedImageDataset

    dataroot_gt: {LOCAL_VAL_CLEAN}
    dataroot_lq: {LOCAL_VAL_NOISY}
    filename_tmpl: '{{}}'

    io_backend:
      type: disk

network_g:
  type: MambaIRv2
  upscale: 1
  in_chans: {IN_CHANS}
  img_size: 512
  img_range: 1.0
  embed_dim: {EMBED_DIM}
  d_state: 8
  depths: {DEPTHS}
  num_heads: {NUM_HEADS}
  window_size: 16
  inner_rank: 32
  num_tokens: 64
  convffn_kernel_size: 5
  mlp_ratio: 2
  upsampler: ''
  resi_connection: '1conv'

path:
  pretrain_network_g: {CHECKPOINT_PATH}

val:
  save_img: true
  suffix: ~

  metrics:
    psnr:
      type: calculate_psnr
      crop_border: 0
      test_y_channel: false
    ssim:
      type: calculate_ssim
      crop_border: 0
      test_y_channel: false
"""

with open(TEST_YML_PATH, "w") as fh:
    fh.write(TEST_YML)

print(f"Test config written to:")
print(f"  {TEST_YML_PATH}")
print(f"\nKey settings:")
print(f"  model_type = {MODEL_TYPE}")
print(f"  in_chans   = {IN_CHANS}")
print(f"  img_size   = 512  (full resolution)")
print(f"  checkpoint = {CHECKPOINT_PATH}")


## 8  Run Paired Evaluation with `basicsr/test.py`

Evaluates the model on the validation set and reports per-image and average PSNR/SSIM.
Denoised images are saved under `results/MambaIRv2_TPAF_Denoise_Test/`.

In [ ]:
import os
os.chdir(MAMBAIR_DIR)

print(f"Running evaluation with: {TEST_YML_PATH}\n")
!python basicsr/test.py -opt {TEST_YML_PATH}

## 9  Inference Helper Functions

Three functions:
- `load_model()` — loads the MambaIRv2 checkpoint
- `denoise_image()` — full-image inference
- `denoise_tiled()` — tiled inference (OOM fallback; overlapping tiles are blended)

In [ ]:
import os, sys, glob
import numpy as np
import torch
from PIL import Image

sys.path.insert(0, MAMBAIR_DIR)
from basicsr.archs.mambairv2_arch import MambaIRv2


def load_model(checkpoint_path: str, in_chans: int, device: str = "cuda") -> MambaIRv2:
    """Load a trained MambaIRv2 checkpoint."""
    model = MambaIRv2(
        upscale          = 1,
        in_chans         = in_chans,
        img_size         = 512,     # full-resolution at inference
        img_range        = 1.0,
        embed_dim        = EMBED_DIM,
        d_state          = 8,
        depths           = DEPTHS,
        num_heads        = NUM_HEADS,
        window_size      = 16,
        inner_rank       = 32,
        num_tokens       = 64,
        convffn_kernel_size = 5,
        mlp_ratio        = 2,
        upsampler        = '',
        resi_connection  = '1conv',
    )
    sd = torch.load(checkpoint_path, map_location=device)
    # Handle different checkpoint formats
    if isinstance(sd, dict):
        sd = sd.get("params_ema", sd.get("params", sd))
    model.load_state_dict(sd, strict=True)
    model.to(device).eval()
    return model


def _img_to_tensor(img_path: str, in_chans: int, device: str) -> torch.Tensor:
    """Load a grayscale image as a float32 tensor of shape (1, C, H, W)."""
    img = Image.open(img_path).convert("L")
    arr = np.array(img, dtype=np.float32) / 255.0  # (H, W)
    if in_chans == 3:
        arr = np.stack([arr, arr, arr], axis=0)     # (3, H, W)
    else:
        arr = arr[np.newaxis]                        # (1, H, W)
    return torch.from_numpy(arr).unsqueeze(0).to(device)  # (1, C, H, W)


def _tensor_to_uint8(tensor: torch.Tensor, in_chans: int) -> np.ndarray:
    """Convert model output tensor to uint8 numpy array (grayscale)."""
    out = tensor.squeeze()           # (C, H, W) or (H, W)
    if in_chans == 3 and out.ndim == 3:
        out = out.mean(dim=0)        # average 3 channels back to grayscale
    return (out.clamp(0.0, 1.0).cpu().numpy() * 255.0).round().astype(np.uint8)


def denoise_image(model: MambaIRv2, img_path: str,
                  in_chans: int, device: str = "cuda") -> np.ndarray:
    """Full-image inference. May raise torch.cuda.OutOfMemoryError on large images."""
    tensor = _img_to_tensor(img_path, in_chans, device)
    with torch.no_grad():
        output = model(tensor)
    return _tensor_to_uint8(output, in_chans)


def denoise_tiled(model: MambaIRv2, img_path: str, in_chans: int,
                  tile_size: int = TILE_SIZE, overlap: int = TILE_OVERLAP,
                  device: str = "cuda") -> np.ndarray:
    """
    Tiled inference with overlap blending (OOM fallback).
    Overlapping regions are averaged, which smooths tile-boundary artefacts.
    tile_size must be divisible by window_size (16).
    """
    assert tile_size % 16 == 0, f"tile_size must be divisible by 16, got {tile_size}"
    img_tensor = _img_to_tensor(img_path, in_chans, device)
    _, _, H, W = img_tensor.shape

    output = torch.zeros_like(img_tensor)
    weight = torch.zeros_like(img_tensor)
    stride = tile_size - overlap

    for y in range(0, H, stride):
        for x in range(0, W, stride):
            y_end   = min(y + tile_size, H)
            x_end   = min(x + tile_size, W)
            y_start = y_end - tile_size    # ensure tile is always tile_size×tile_size
            x_start = x_end - tile_size

            tile = img_tensor[:, :, y_start:y_end, x_start:x_end]
            with torch.no_grad():
                tile_out = model(tile)

            output[:, :, y_start:y_end, x_start:x_end] += tile_out
            weight[:, :, y_start:y_end, x_start:x_end] += 1.0

    return _tensor_to_uint8(output / weight, in_chans)


print("Inference helper functions defined.")

## 10  Load Model

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

model = load_model(CHECKPOINT_PATH, IN_CHANS, DEVICE)
print(f"Model loaded successfully.")
print(f"  in_chans   = {IN_CHANS}")
print(f"  checkpoint = {CHECKPOINT_PATH}")

## 11  Standalone Inference on Unpaired Noisy Images

Processes all images in `LOCAL_NOISY_DIR` (copied from `DRIVE_NOISY_DIR`).
Falls back to **tiled inference** automatically if a full-image forward pass runs out of GPU memory.

In [ ]:
import glob, os
from pathlib import Path

STANDALONE_OUT = f"{LOCAL_RESULTS_DIR}/standalone"
os.makedirs(STANDALONE_OUT, exist_ok=True)

if LOCAL_NOISY_DIR is None or not os.path.exists(LOCAL_NOISY_DIR):
    print("LOCAL_NOISY_DIR not set or missing — skipping standalone inference.")
    print("Set DRIVE_NOISY_DIR in the configuration cell and re-run cell 3.")
else:
    EXTS = {".png", ".tif", ".tiff", ".jpg", ".jpeg", ".bmp"}
    img_paths = sorted(
        p for p in glob.glob(f"{LOCAL_NOISY_DIR}/**/*", recursive=True)
        if Path(p).suffix.lower() in EXTS
    )
    print(f"Found {len(img_paths)} image(s) in {LOCAL_NOISY_DIR}")

    for i, img_path in enumerate(img_paths):
        fname = os.path.basename(img_path)
        try:
            out = denoise_image(model, img_path, IN_CHANS, DEVICE)
        except (RuntimeError, torch.cuda.OutOfMemoryError) as oom_err:
            if "out of memory" in str(oom_err).lower() or isinstance(oom_err, torch.cuda.OutOfMemoryError):
                print(f"  OOM on full image — switching to tiled inference: {fname}")
                torch.cuda.empty_cache()
                out = denoise_tiled(model, img_path, IN_CHANS, TILE_SIZE, TILE_OVERLAP, DEVICE)
            else:
                raise

        save_path = os.path.join(STANDALONE_OUT, Path(fname).stem + ".png")
        Image.fromarray(out, mode="L").save(save_path)

        if (i + 1) % 10 == 0 or (i + 1) == len(img_paths):
            print(f"  [{i+1:>4}/{len(img_paths)}] saved: {fname}")

    print(f"\nStandalone inference complete. Results in: {STANDALONE_OUT}")

## 12  Visual Side-by-Side Comparison

Shows `VIS_NUM_IMAGES` randomly selected validation images:
**Noisy input → Model output → Ground truth**

In [ ]:
import matplotlib.pyplot as plt
import random, os
import numpy as np
from PIL import Image

random.seed(42)
val_fnames = sorted(f for f in os.listdir(LOCAL_VAL_CLEAN) if not f.startswith("."))
idxs = random.sample(range(len(val_fnames)), min(VIS_NUM_IMAGES, len(val_fnames)))

n_rows = len(idxs)
fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows), squeeze=False)

for row, idx in enumerate(idxs):
    fname      = val_fnames[idx]
    noisy_path = os.path.join(LOCAL_VAL_NOISY, fname)
    clean_path = os.path.join(LOCAL_VAL_CLEAN,  fname)

    noisy_arr = np.array(Image.open(noisy_path).convert("L"))
    clean_arr = np.array(Image.open(clean_path).convert("L"))

    # Run inference (tiled fallback if needed)
    try:
        denoised_arr = denoise_image(model, noisy_path, IN_CHANS, DEVICE)
    except (RuntimeError, torch.cuda.OutOfMemoryError) as e:
        if "out of memory" in str(e).lower() or isinstance(e, torch.cuda.OutOfMemoryError):
            torch.cuda.empty_cache()
            denoised_arr = denoise_tiled(model, noisy_path, IN_CHANS,
                                         TILE_SIZE, TILE_OVERLAP, DEVICE)
        else:
            raise

    for col, (arr, title) in enumerate([
        (noisy_arr,   "Noisy Input"),
        (denoised_arr,"Denoised (MambaIRv2)"),
        (clean_arr,   "Ground Truth"),
    ]):
        ax = axes[row, col]
        ax.imshow(arr, cmap="gray", vmin=0, vmax=255)
        ax.set_title(title, fontsize=12)
        ax.axis("off")

plt.suptitle("MambaIRv2 TPAF Denoising — Validation Samples", fontsize=14, y=1.01)
plt.tight_layout()

comp_path = os.path.join(LOCAL_RESULTS_DIR, "comparison.png")
plt.savefig(comp_path, dpi=100, bbox_inches="tight")
plt.show()
print(f"Comparison figure saved to: {comp_path}")

## 13  Aggregate Metrics (Phase 8)

Computes PSNR and SSIM for:
- **Noisy vs Clean** — baseline (how much noise there is)
- **Denoised vs Clean** — model performance

Results are also written to a CSV file.

In [ ]:
import os, csv
import numpy as np
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity as calc_ssim

val_fnames = sorted(f for f in os.listdir(LOCAL_VAL_CLEAN) if not f.startswith("."))

# We'll denoise each val image on-the-fly for metrics
# (re-uses the already-loaded model)

rows = []
psnrs_noisy, ssims_noisy       = [], []
psnrs_denoised, ssims_denoised = [], []

print(f"Computing metrics on {len(val_fnames)} validation pairs ...")

for i, fname in enumerate(val_fnames):
    clean_arr = np.array(Image.open(os.path.join(LOCAL_VAL_CLEAN, fname)).convert("L"))
    noisy_arr = np.array(Image.open(os.path.join(LOCAL_VAL_NOISY, fname)).convert("L"))

    try:
        den_arr = denoise_image(model, os.path.join(LOCAL_VAL_NOISY, fname),
                                IN_CHANS, DEVICE)
    except (RuntimeError, torch.cuda.OutOfMemoryError) as e:
        if "out of memory" in str(e).lower() or isinstance(e, torch.cuda.OutOfMemoryError):
            torch.cuda.empty_cache()
            den_arr = denoise_tiled(model, os.path.join(LOCAL_VAL_NOISY, fname),
                                    IN_CHANS, TILE_SIZE, TILE_OVERLAP, DEVICE)
        else:
            raise

    pn = calc_psnr(clean_arr, noisy_arr,   data_range=255)
    sn = calc_ssim(clean_arr, noisy_arr,   data_range=255)
    pd = calc_psnr(clean_arr, den_arr,     data_range=255)
    sd = calc_ssim(clean_arr, den_arr,     data_range=255)

    psnrs_noisy.append(pn);   ssims_noisy.append(sn)
    psnrs_denoised.append(pd); ssims_denoised.append(sd)
    rows.append({"filename": fname,
                 "psnr_noisy": pn, "ssim_noisy": sn,
                 "psnr_denoised": pd, "ssim_denoised": sd})

    if (i + 1) % 50 == 0 or (i + 1) == len(val_fnames):
        print(f"  [{i+1}/{len(val_fnames)}]  "
              f"running avg denoised PSNR: {np.mean(psnrs_denoised):.2f} dB")

# Save CSV
csv_path = os.path.join(LOCAL_RESULTS_DIR, "metrics.csv")
with open(csv_path, "w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

# Summary table
print()
print(f"{'Metric':<20} {'Noisy Input':>15} {'Denoised':>15}  {'Improvement':>12}")
print("─" * 65)
print(f"{'PSNR (dB)':<20} {np.mean(psnrs_noisy):>15.4f} "
      f"{np.mean(psnrs_denoised):>15.4f}  "
      f"{np.mean(psnrs_denoised)-np.mean(psnrs_noisy):>+11.4f}")
print(f"{'SSIM':<20} {np.mean(ssims_noisy):>15.4f} "
      f"{np.mean(ssims_denoised):>15.4f}  "
      f"{np.mean(ssims_denoised)-np.mean(ssims_noisy):>+11.4f}")
print()
print(f"Per-image metrics saved to: {csv_path}")

## 14  Copy All Results to Google Drive

In [ ]:
import shutil, os

shutil.copytree(LOCAL_RESULTS_DIR, DRIVE_RESULTS_DIR, dirs_exist_ok=True)
print(f"Results copied to Drive: {DRIVE_RESULTS_DIR}")
print("Contents:")
for root, dirs, files in os.walk(DRIVE_RESULTS_DIR):
    level = root.replace(DRIVE_RESULTS_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files[:5]:
            print(f"{indent}  {f}")
        if len(files) > 5:
            print(f"{indent}  ... ({len(files) - 5} more files)")